In [2]:
from openpyxl import load_workbook
from tqdm import tqdm
import csv
import os
import ctypes
from openpyxl import load_workbook
import pandas as pd

patient_df = pd.read_csv(r"\\pesamba.int.chpc.utah.edu\PE-projects3-1\proj_tnx_ops\Proj_Caitlin Roake\patient_filtered.csv", usecols=["patient_id"])
patient_ids = set(patient_df["patient_id"].dropna())


file_path = r"\\pesamba.int.chpc.utah.edu\PE-projects3-1\proj_tnx_ops\Proj_Caitlin Roake\vitals_signs.csv"
file_size = os.path.getsize(file_path)


chunksize = 1_000_000
output_file = "filtered_vs.csv"
first_write = True

with tqdm(total=file_size, unit="B", unit_scale=True, desc="Processing vitals_signs") as pbar:
    with open(file_path, "rb") as f:
        reader = pd.read_csv(
            f,
            chunksize=chunksize,
            dtype={"patient_id": str}
        )

        for chunk in reader:
            matched = chunk[chunk["patient_id"].isin(patient_ids)]

            if not matched.empty:
                matched.to_csv(
                    output_file,
                    mode="w" if first_write else "a",
                    header=first_write,
                    index=False,
                )
                first_write = False

            # Update progress bar
            pbar.update(f.tell() - pbar.n)

Processing vitals_signs: 100%|██████████| 133G/133G [1:18:09<00:00, 28.3MB/s] 
